# GMX Cross-Exchange Validation
## Comparing GMX Pool Liquidity & Open Interest against Binance and Hyperliquid Volume

This notebook validates whether ebb and flow patterns on GMX (open interest, pool liquidity) 
correlate with trading activity on centralised exchanges (Binance, Hyperliquid).

**Data sources:**
- Binance futures 1h OHLCV (via feather files)
- Hyperliquid futures 1h OHLCV (via feather files)  
- GMX V2 pool liquidity (PoolAmountUpdated events, daily snapshots)
- GMX V2 open interest (OpenInterestUpdated events, daily values)

**Analysis:**
1. Rank all GMX-compatible markets by Binance/HL 30d rolling volume MA
2. Per-market: plot CEX volume alongside GMX OI and pool depth on matching date ranges
3. Scatter: CEX avg volume vs GMX latest OI / pool size (market size relationship)
4. Correlation: how well does CEX volume predict GMX utilisation?

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")

# Absolute data paths
BINANCE_DIR = Path("/Users/avik/Work/tradingstrategy/my-strategies/user_data/data/binance/futures")
HL_DIR = Path("/Users/avik/Work/tradingstrategy/my-strategies/user_data/data/hyperliquid/futures")
GMX_OI_DIR = Path(
    "/Users/avik/Work/tradingstrategy/gmx_historical_data/user_data/data/gmx/open_interest/arbitrum/raw"
)
GMX_POOL_DIR = Path(
    "/Users/avik/Work/tradingstrategy/gmx_historical_data/user_data/data/gmx/pool_liquidity/arbitrum/snapshots"
)

TEMPLATE = "plotly_white"
SCALE_30 = 10**30
MA_WINDOW = 30  # days for rolling volume average


def _normalize_base(base: str) -> str:
    """Strip exchange-specific multiplier prefixes.

    Binance uses ``1000BONK``, Hyperliquid uses ``KBONK``; both map to ``BONK``.
    """
    base = re.sub(r"^1000", "", base)
    base = re.sub(r"^K(?=[A-Z])", "", base)
    return base


# ── Auto-discover bases from feather files ────────────────────────────────────
# Build raw-name → normalised-name maps so load functions can find the right file.
_bin_raw_bases = sorted(
    {f.name.split("_USDT_USDT")[0] for f in BINANCE_DIR.glob("*_USDT_USDT-1h-futures.feather")}
)
_hl_raw_bases = sorted(
    {f.name.split("_USDC_USDC")[0] for f in HL_DIR.glob("*_USDC_USDC-1h-futures.feather")}
)

# norm → raw  (for looking up files by normalised name)
_bin_norm_to_raw: dict[str, str] = {_normalize_base(b): b for b in _bin_raw_bases}
_hl_norm_to_raw: dict[str, str] = {_normalize_base(b): b for b in _hl_raw_bases}

_binance_bases_norm = sorted(_bin_norm_to_raw.keys())
_hl_bases_norm = sorted(_hl_norm_to_raw.keys())

# GMX market universe (all directories with OI data)
_gmx_all_bases = sorted(
    {
        d.name.split("_USD")[0]
        for d in GMX_OI_DIR.iterdir()
        if d.is_dir() and (d / "data.parquet").exists()
    }
)

# All normalised CEX bases (union)
_all_cex_bases_norm = sorted(set(_binance_bases_norm) | set(_hl_bases_norm))

# Intersection (bases on BOTH exchanges — for charts that need both)
COMMON_BASES = sorted(set(_binance_bases_norm) & set(_hl_bases_norm))

print(
    f"Binance 1h futures:     {len(_bin_raw_bases)} pairs  →  {len(_binance_bases_norm)} normalised bases"
)
print(
    f"Hyperliquid 1h futures: {len(_hl_raw_bases)} pairs  →  {len(_hl_bases_norm)} normalised bases"
)
print(f"GMX markets (all):      {len(_gmx_all_bases)} unique bases")
print(f"CEX union (either):     {len(_all_cex_bases_norm)} bases")
print(f"CEX intersection:       {len(COMMON_BASES)} bases")
gmx_no_cex = sorted(set(_gmx_all_bases) - set(_all_cex_bases_norm))
print(f"GMX-only (no CEX data): {gmx_no_cex}")

Binance 1h futures:     96 pairs  →  96 normalised bases
Hyperliquid 1h futures: 93 pairs  →  93 normalised bases
GMX markets (all):      109 unique bases
CEX union (either):     97 bases
CEX intersection:       92 bases
GMX-only (no CEX data): ['AI16Z', 'CRO', 'KAS', 'KTA', 'MKR', 'OKB', 'OM', 'PI', 'SPX6900', 'WELL', 'XAUT.v2', 'XAUT_(deprecated)']


In [2]:
def load_cex_volume(
    data_dir: Path,
    norm_to_raw: dict[str, str],
    quote_suffix: str,
    bases: list[str],
) -> pd.DataFrame:
    """Load CEX 1h futures feather files and aggregate to daily USD volume.

    :param data_dir: Directory containing feather files.
    :param norm_to_raw: Mapping from normalised base (e.g. ``BONK``) to raw
        filename base (e.g. ``1000BONK`` or ``KBONK``).
    :param quote_suffix: Filename suffix after the raw base,
        e.g. ``"_USDT_USDT-1h-futures.feather"``.
    :param bases: Normalised base symbols to load.
    :return: DataFrame with columns date, usd_volume, base (normalised).
    """
    frames = []
    for base_norm in bases:
        raw = norm_to_raw.get(base_norm)
        if raw is None:
            continue
        f = data_dir / f"{raw}{quote_suffix}"
        if not f.exists():
            continue
        try:
            df = pd.read_feather(f)
            df["date"] = pd.to_datetime(df["date"], utc=True).dt.normalize()
            df["usd_volume"] = df["close"] * df["volume"]
            daily = df.groupby("date")["usd_volume"].sum().reset_index()
            daily["base"] = base_norm
            frames.append(daily)
        except Exception as e:
            print(f"  Skip {f.name}: {e}")
    if not frames:
        return pd.DataFrame(columns=["date", "usd_volume", "base"])
    return pd.concat(frames, ignore_index=True)


print("Loading Binance volume (all pairs)...")
binance_vol = load_cex_volume(
    BINANCE_DIR, _bin_norm_to_raw, "_USDT_USDT-1h-futures.feather", _binance_bases_norm
)
print(f"  Loaded {binance_vol['base'].nunique()} markets, {len(binance_vol):,} daily rows")
print(f"  Date range: {binance_vol['date'].min().date()} → {binance_vol['date'].max().date()}")

print("Loading Hyperliquid volume (all pairs)...")
hl_vol = load_cex_volume(HL_DIR, _hl_norm_to_raw, "_USDC_USDC-1h-futures.feather", _hl_bases_norm)
print(f"  Loaded {hl_vol['base'].nunique()} markets, {len(hl_vol):,} daily rows")
print(f"  Date range: {hl_vol['date'].min().date()} → {hl_vol['date'].max().date()}")

Loading Binance volume (all pairs)...
  Skip XPL_USDT_USDT-1h-futures.feather: Not an Arrow file
  Skip ZEC_USDT_USDT-1h-futures.feather: Not an Arrow file
  Skip ZORA_USDT_USDT-1h-futures.feather: Not an Arrow file
  Loaded 93 markets, 75,624 daily rows
  Date range: 2022-01-01 → 2026-03-11
Loading Hyperliquid volume (all pairs)...
  Loaded 93 markets, 43,200 daily rows
  Date range: 2021-01-01 → 2026-03-11


In [3]:
def add_rolling_ma(vol_df: pd.DataFrame, window: int = 30) -> pd.DataFrame:
    """Add rolling MA volume column sorted by base and date.

    :param vol_df: DataFrame with columns base, date, usd_volume.
    :param window: Rolling window size in days.
    :return: DataFrame with additional vol_ma{window}d column.
    """
    vol_df = vol_df.sort_values(["base", "date"])
    vol_df[f"vol_ma{window}d"] = vol_df.groupby("base")["usd_volume"].transform(
        lambda s: s.rolling(window, min_periods=max(1, window // 2)).mean()
    )
    return vol_df


binance_vol = add_rolling_ma(binance_vol, MA_WINDOW)
hl_vol = add_rolling_ma(hl_vol, MA_WINDOW)

# Rank by average 30d MA over the full dataset
binance_rank = (
    binance_vol.groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={f"vol_ma{MA_WINDOW}d": "avg_ma_vol"})
)
binance_rank["binance_rank"] = range(1, len(binance_rank) + 1)

hl_rank = (
    hl_vol.groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={f"vol_ma{MA_WINDOW}d": "avg_ma_vol"})
)
hl_rank["hl_rank"] = range(1, len(hl_rank) + 1)

print("Binance top 10 by avg 30d MA volume (USD/day):")
print(binance_rank.head(10).to_string(index=False))
print("\nHyperliquid top 10 by avg 30d MA volume (USD/day):")
print(hl_rank.head(10).to_string(index=False))

Binance top 10 by avg 30d MA volume (USD/day):
 base   avg_ma_vol  binance_rank
  BTC 1.705428e+10             1
  ETH 1.212589e+10             2
  SOL 3.589849e+09             3
 DOGE 1.441230e+09             4
  XRP 1.202866e+09             5
 PEPE 1.003660e+09             6
  SUI 6.496965e+08             7
ASTER 6.119900e+08             8
TRUMP 6.103164e+08             9
  BNB 5.581728e+08            10

Hyperliquid top 10 by avg 30d MA volume (USD/day):
base   avg_ma_vol  hl_rank
 BTC 1.525542e+10        1
 ETH 9.055839e+09        2
 SOL 2.100416e+09        3
DOGE 1.118441e+09        4
PEPE 8.824855e+08        5
 SUI 5.455107e+08        6
 WIF 4.918297e+08        7
HYPE 4.463105e+08        8
 ENA 4.159651e+08        9
AVAX 3.821531e+08       10


In [4]:
def load_gmx_oi(oi_dir: Path, bases: list[str]) -> pd.DataFrame:
    """Load GMX OI parquet files and return daily total OI per base.

    :param oi_dir: Directory containing per-symbol OI parquet files.
    :param bases: List of base asset symbols to include.
    :return: DataFrame with columns date, base, oi_usd (daily end-of-day OI).
    """
    frames = []
    if not oi_dir.exists():
        print(f"  OI dir not found: {oi_dir}")
        return pd.DataFrame()

    for symbol_dir in sorted(oi_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "data.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            # Extract base from symbol (e.g. "BTC/USD [BTC-WBTC]" → "BTC")
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            if sym_clean not in bases:
                continue
            # Convert 30-decimal USD values
            if "next_value_usd" in df.columns:
                df["oi_usd"] = pd.to_numeric(df["next_value_usd"], errors="coerce") / SCALE_30
            elif "nextValueUsd" in df.columns:
                df["oi_usd"] = pd.to_numeric(df["nextValueUsd"], errors="coerce") / SCALE_30
            else:
                continue
            # Timestamp column
            ts_col = "block_timestamp" if "block_timestamp" in df.columns else "blockTimestamp"
            df["ts"] = pd.to_datetime(df[ts_col], unit="s", utc=True)
            df["date"] = df["ts"].dt.normalize()
            df["base"] = sym_clean
            # Daily end-of-day OI (last value per day)
            daily = df.sort_values("ts").groupby(["date", "base"])["oi_usd"].last().reset_index()
            frames.append(daily)
        except Exception as e:
            print(f"  Skip {symbol_dir.name}: {e}")

    if not frames:
        return pd.DataFrame(columns=["date", "base", "oi_usd"])
    result = pd.concat(frames, ignore_index=True)
    # Sum across multiple markets for same base (e.g. BTC has BTC-WBTC and BTC-USDC markets)
    return result.groupby(["date", "base"])["oi_usd"].sum().reset_index()


print("Loading GMX OI...")
gmx_oi = load_gmx_oi(GMX_OI_DIR, COMMON_BASES)
if not gmx_oi.empty:
    print(f"  Loaded {gmx_oi['base'].nunique()} GMX OI markets, {len(gmx_oi)} daily rows")
    print(f"  Date range: {gmx_oi['date'].min()} → {gmx_oi['date'].max()}")
    print(f"  Markets: {sorted(gmx_oi['base'].unique())}")
else:
    print("  No GMX OI data found (run make oi first)")

Loading GMX OI...
  Loaded 92 GMX OI markets, 38174 daily rows
  Date range: 2023-08-10 00:00:00+00:00 → 2026-03-10 00:00:00+00:00
  Markets: ['0G', 'AAVE', 'ADA', 'AERO', 'AIXBT', 'ALGO', 'ANIME', 'APE', 'APT', 'AR', 'ARB', 'ASTER', 'ATOM', 'AVAX', 'AVNT', 'BCH', 'BERA', 'BNB', 'BOME', 'BONK', 'BRETT', 'BTC', 'CAKE', 'CC', 'CRV', 'DASH', 'DOGE', 'DOT', 'DYDX', 'EIGEN', 'ENA', 'ETH', 'FARTCOIN', 'FET', 'FIL', 'FLOKI', 'GMX', 'HBAR', 'HYPE', 'ICP', 'INJ', 'IP', 'JTO', 'JUP', 'LDO', 'LINEA', 'LINK', 'LIT', 'LTC', 'MELANIA', 'MEME', 'MET', 'MEW', 'MON', 'MOODENG', 'MORPHO', 'NEAR', 'ONDO', 'OP', 'ORDI', 'PENDLE', 'PENGU', 'PEPE', 'POL', 'PUMP', 'RENDER', 'S', 'SEI', 'SHIB', 'SKY', 'SOL', 'STX', 'SUI', 'SYRUP', 'TAO', 'TIA', 'TON', 'TRUMP', 'TRX', 'UNI', 'VIRTUAL', 'VVV', 'WIF', 'WLD', 'WLFI', 'XLM', 'XMR', 'XPL', 'XRP', 'ZEC', 'ZORA', 'ZRO']


In [5]:
# USD-pegged stablecoins on Arbitrum (used as USD proxy for pool depth)
_STABLE_TOKENS = {
    "0xaf88d065e77c8cc2239327c5edb3a432268e5831",  # USDC (native)
    "0xff970a61a04b1ca14834a43f5de4533ebddb5cc8",  # USDC.e (bridged)
    "0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9",  # USDT
    "0xda10009cbd5d07dd0cecc66161fc93d7c9000da1",  # DAI
}


def load_gmx_pool(pool_dir: Path, bases: list[str]) -> pd.DataFrame:
    """Load GMX daily pool liquidity snapshots, returning stable-token USD proxy per base.

    Pool tokens are in native token units (BTC, ETH, etc.) so we use the stablecoin
    leg (USDC/USDT) as a USD proxy for the short side.  Non-stable token pools are
    retained as a separate ``pool_tokens_long`` column for trend analysis.

    :param pool_dir: Directory containing per-symbol ``daily.parquet`` files.
    :param bases: List of base asset symbols to include (e.g. ``["BTC", "ETH"]``).
    :return: DataFrame with columns date, base, pool_usd_proxy, pool_tokens_long.
    """
    frames = []
    if not pool_dir.exists():
        print(f"  Pool dir not found: {pool_dir}")
        return pd.DataFrame()

    for symbol_dir in sorted(pool_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "daily.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            if sym_clean not in bases:
                continue
            df["date"] = pd.to_datetime(df["date"], utc=True)
            df["base"] = sym_clean
            df["token_lc"] = df["token"].str.lower()
            # Stablecoin tokens → USD proxy
            stable = (
                df[df["token_lc"].isin(_STABLE_TOKENS)]
                .groupby(["date", "base"])["pool_tokens"]
                .sum()
                .reset_index()
                .rename(columns={"pool_tokens": "pool_usd_proxy"})
            )
            # Non-stable tokens → long-side depth (in native token units)
            long_tok = (
                df[~df["token_lc"].isin(_STABLE_TOKENS)]
                .groupby(["date", "base"])["pool_tokens"]
                .sum()
                .reset_index()
                .rename(columns={"pool_tokens": "pool_tokens_long"})
            )
            merged = stable.merge(long_tok, on=["date", "base"], how="outer")
            frames.append(merged)
        except Exception as e:
            print(f"  Skip {symbol_dir.name}: {e}")

    if not frames:
        return pd.DataFrame(columns=["date", "base", "pool_usd_proxy", "pool_tokens_long"])
    result = pd.concat(frames, ignore_index=True)
    # Sum across multiple market variants for the same base
    return (
        result.groupby(["date", "base"])[["pool_usd_proxy", "pool_tokens_long"]].sum().reset_index()
    )


print("Loading GMX Pool Liquidity...")
gmx_pool = load_gmx_pool(GMX_POOL_DIR, COMMON_BASES)
if not gmx_pool.empty:
    print(f"  Loaded {gmx_pool['base'].nunique()} GMX pool markets, {len(gmx_pool)} daily rows")
    print(f"  Date range: {gmx_pool['date'].min()} → {gmx_pool['date'].max()}")
    print(f"  Markets: {sorted(gmx_pool['base'].unique())}")
else:
    print("  No GMX pool data found (run make pool-liquidity first)")

Loading GMX Pool Liquidity...
  Loaded 92 GMX pool markets, 38813 daily rows
  Date range: 2023-08-10 00:00:00+00:00 → 2026-03-10 00:00:00+00:00
  Markets: ['0G', 'AAVE', 'ADA', 'AERO', 'AIXBT', 'ALGO', 'ANIME', 'APE', 'APT', 'AR', 'ARB', 'ASTER', 'ATOM', 'AVAX', 'AVNT', 'BCH', 'BERA', 'BNB', 'BOME', 'BONK', 'BRETT', 'BTC', 'CAKE', 'CC', 'CRV', 'DASH', 'DOGE', 'DOT', 'DYDX', 'EIGEN', 'ENA', 'ETH', 'FARTCOIN', 'FET', 'FIL', 'FLOKI', 'GMX', 'HBAR', 'HYPE', 'ICP', 'INJ', 'IP', 'JTO', 'JUP', 'LDO', 'LINEA', 'LINK', 'LIT', 'LTC', 'MELANIA', 'MEME', 'MET', 'MEW', 'MON', 'MOODENG', 'MORPHO', 'NEAR', 'ONDO', 'OP', 'ORDI', 'PENDLE', 'PENGU', 'PEPE', 'POL', 'PUMP', 'RENDER', 'S', 'SEI', 'SHIB', 'SKY', 'SOL', 'STX', 'SUI', 'SYRUP', 'TAO', 'TIA', 'TON', 'TRUMP', 'TRX', 'UNI', 'VIRTUAL', 'VVV', 'WIF', 'WLD', 'WLFI', 'XLM', 'XMR', 'XPL', 'XRP', 'ZEC', 'ZORA', 'ZRO']


In [6]:
def build_combined(binance_vol, hl_vol, gmx_oi, gmx_pool, ma_window=30):
    """Merge all data sources into a single aligned DataFrame."""
    col = f"vol_ma{ma_window}d"
    b = binance_vol[["date", "base", col]].rename(columns={col: "binance_vol_ma"})
    h = hl_vol[["date", "base", col]].rename(columns={col: "hl_vol_ma"})

    dfs = [b.set_index(["date", "base"]), h.set_index(["date", "base"])]
    if not gmx_oi.empty:
        dfs.append(gmx_oi.set_index(["date", "base"]))
    if not gmx_pool.empty:
        pool_cols = ["pool_usd_proxy"] + (
            ["pool_tokens_long"] if "pool_tokens_long" in gmx_pool.columns else []
        )
        dfs.append(gmx_pool[["date", "base"] + pool_cols].set_index(["date", "base"]))

    combined = pd.concat(dfs, axis=1).reset_index()
    return combined.sort_values(["base", "date"])


combined = build_combined(binance_vol, hl_vol, gmx_oi, gmx_pool, MA_WINDOW)

has_binance = set(binance_vol["base"].unique())
has_hl = set(hl_vol["base"].unique())
has_gmx_oi = set(gmx_oi["base"].unique()) if not gmx_oi.empty else set()
has_gmx_pool = set(gmx_pool["base"].unique()) if not gmx_pool.empty else set()

# UNIVERSE: all GMX markets that have data from at least one CEX
# (97 markets — only the 12 GMX-exclusive tokens have no CEX data)
UNIVERSE = sorted(set(_gmx_all_bases) & (has_binance | has_hl))

universe_all4 = sorted(has_binance & has_hl & has_gmx_oi & has_gmx_pool)
gmx_no_cex_data = sorted(set(_gmx_all_bases) - (has_binance | has_hl))

print(f"Binance bases:            {len(has_binance)}")
print(f"Hyperliquid bases:        {len(has_hl)}")
print(f"GMX OI bases:             {len(has_gmx_oi)}")
print(f"GMX Pool bases:           {len(has_gmx_pool)}")
print(f"\nUNIVERSE (GMX + any CEX): {len(UNIVERSE)} markets")
print(f"All 4 sources:            {len(universe_all4)} markets")
print(f"GMX-only (no CEX):        {gmx_no_cex_data}")

Binance bases:            93
Hyperliquid bases:        93
GMX OI bases:             92
GMX Pool bases:           92

UNIVERSE (GMX + any CEX): 97 markets
All 4 sources:            89 markets
GMX-only (no CEX):        ['AI16Z', 'CRO', 'KAS', 'KTA', 'MKR', 'OKB', 'OM', 'PI', 'SPX6900', 'WELL', 'XAUT.v2', 'XAUT_(deprecated)']


In [7]:
latest_b = (
    binance_vol[binance_vol["base"].isin(UNIVERSE)]
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
latest_b.columns = ["base", "avg_vol"]

fig = px.bar(
    latest_b,
    x="base",
    y="avg_vol",
    title=f"GMX-Compatible Markets — Binance {MA_WINDOW}d Rolling Avg Volume (USD/day)",
    labels={"avg_vol": "Avg Daily Volume (USD)", "base": "Market"},
    template=TEMPLATE,
    color="avg_vol",
    color_continuous_scale="Blues",
)
fig.update_layout(showlegend=False, xaxis_tickangle=-45)
fig.show()

In [8]:
latest_h = (
    hl_vol[hl_vol["base"].isin(UNIVERSE)]
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
latest_h.columns = ["base", "avg_vol"]

fig = px.bar(
    latest_h,
    x="base",
    y="avg_vol",
    title=f"GMX-Compatible Markets — Hyperliquid {MA_WINDOW}d Rolling Avg Volume (USD/day)",
    labels={"avg_vol": "Avg Daily Volume (USD)", "base": "Market"},
    template=TEMPLATE,
    color="avg_vol",
    color_continuous_scale="Purples",
)
fig.update_layout(showlegend=False, xaxis_tickangle=-45)
fig.show()

In [9]:
merged_cex = (
    binance_vol[binance_vol["base"].isin(UNIVERSE)]
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .rename("binance_avg")
    .to_frame()
    .join(
        hl_vol[hl_vol["base"].isin(UNIVERSE)]
        .groupby("base")[f"vol_ma{MA_WINDOW}d"]
        .mean()
        .rename("hl_avg")
    )
    .dropna()
    .reset_index()
)

fig = px.scatter(
    merged_cex,
    x="binance_avg",
    y="hl_avg",
    text="base",
    title=f"Binance vs Hyperliquid — Avg {MA_WINDOW}d Volume (USD/day)",
    labels={"binance_avg": "Binance (USD/day)", "hl_avg": "Hyperliquid (USD/day)"},
    template=TEMPLATE,
    trendline="ols",
    trendline_color_override="orange",
)
fig.update_traces(textposition="top center", marker_size=10)
fig.show()

In [10]:
top_bases = (
    binance_vol[binance_vol["base"].isin(UNIVERSE)]
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .sort_values(ascending=False)
    .head(9)
    .index.tolist()
)
print(f"Plotting top {len(top_bases)} bases by Binance volume: {top_bases}")

for base in top_bases:
    b_df = binance_vol[binance_vol["base"] == base].sort_values("date")
    h_df = hl_vol[hl_vol["base"] == base].sort_values("date")
    oi_df = (
        gmx_oi[gmx_oi["base"] == base].sort_values("date") if not gmx_oi.empty else pd.DataFrame()
    )
    pool_df = (
        gmx_pool[gmx_pool["base"] == base].sort_values("date")
        if not gmx_pool.empty
        else pd.DataFrame()
    )

    has_oi = not oi_df.empty
    has_pool = not pool_df.empty
    n_rows = 1 + int(has_oi) + int(has_pool)
    row_titles = ["CEX Daily Volume (USD)"]
    if has_oi:
        row_titles.append("GMX Open Interest (USD M)")
    if has_pool:
        row_titles.append("GMX Pool Depth — Stablecoin leg (USD M proxy)")

    fig = make_subplots(
        rows=n_rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=row_titles,
        vertical_spacing=0.08,
    )

    fig.add_trace(
        go.Bar(
            x=b_df["date"],
            y=b_df["usd_volume"],
            name="Binance Vol",
            marker_color="#c92eaa",
            opacity=0.35,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=b_df["date"],
            y=b_df[f"vol_ma{MA_WINDOW}d"],
            name=f"Binance {MA_WINDOW}d MA",
            line=dict(color="#1f77b4", width=2),
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=h_df["date"],
            y=h_df[f"vol_ma{MA_WINDOW}d"],
            name=f"HL {MA_WINDOW}d MA",
            line=dict(color="#d62728", width=2, dash="dash"),
        ),
        row=1,
        col=1,
    )

    cur_row = 2
    if has_oi:
        fig.add_trace(
            go.Scatter(
                x=oi_df["date"],
                y=oi_df["oi_usd"] / 1e6,
                name="GMX OI ($M)",
                line=dict(color="#2ca02c", width=2),
                fill="tozeroy",
                fillcolor="rgba(44,160,44,0.1)",
            ),
            row=cur_row,
            col=1,
        )
        fig.update_yaxes(title_text="OI (M USD)", row=cur_row, col=1)
        cur_row += 1

    if has_pool:
        fig.add_trace(
            go.Scatter(
                x=pool_df["date"],
                y=pool_df["pool_usd_proxy"] / 1e6,
                name="GMX Stable Pool ($M)",
                line=dict(color="#9467bd", width=2),
                fill="tozeroy",
                fillcolor="rgba(148,103,189,0.1)",
            ),
            row=cur_row,
            col=1,
        )
        fig.update_yaxes(title_text="Pool ($M)", row=cur_row, col=1)

    fig.update_layout(
        title=f"{base} — CEX Volume vs GMX OI & Pool Liquidity",
        template=TEMPLATE,
        hovermode="x unified",
        height=220 * n_rows + 80,
        showlegend=True,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.show()

Plotting top 9 bases by Binance volume: ['BTC', 'ETH', 'SOL', 'DOGE', 'XRP', 'PEPE', 'SUI', 'ASTER', 'TRUMP']


In [11]:
if not gmx_oi.empty:
    latest_oi = gmx_oi.groupby("base")["oi_usd"].last().reset_index()
    b_avg = (
        binance_vol[binance_vol["base"].isin(UNIVERSE)]
        .groupby("base")[f"vol_ma{MA_WINDOW}d"]
        .mean()
        .reset_index()
    )
    b_avg.columns = ["base", "binance_avg_vol"]

    scatter_df = latest_oi.merge(b_avg, on="base").dropna()
    fig = px.scatter(
        scatter_df,
        x="binance_avg_vol",
        y="oi_usd",
        text="base",
        size="oi_usd",
        title=f"Binance Avg {MA_WINDOW}d Volume vs GMX Latest OI",
        labels={"binance_avg_vol": "Binance Avg Vol (USD/day)", "oi_usd": "GMX OI (USD)"},
        template=TEMPLATE,
        trendline="ols",
        trendline_color_override="orange",
    )
    fig.update_traces(textposition="top center")
    fig.show()

In [12]:
if not gmx_oi.empty:
    h_avg = (
        hl_vol[hl_vol["base"].isin(UNIVERSE)]
        .groupby("base")[f"vol_ma{MA_WINDOW}d"]
        .mean()
        .reset_index()
    )
    h_avg.columns = ["base", "hl_avg_vol"]

    scatter_hl = latest_oi.merge(h_avg, on="base").dropna()
    fig = px.scatter(
        scatter_hl,
        x="hl_avg_vol",
        y="oi_usd",
        text="base",
        size="oi_usd",
        title=f"Hyperliquid Avg {MA_WINDOW}d Volume vs GMX Latest OI",
        labels={"hl_avg_vol": "Hyperliquid Avg Vol (USD/day)", "oi_usd": "GMX OI (USD)"},
        template=TEMPLATE,
        trendline="ols",
        trendline_color_override="steelblue",
    )
    fig.update_traces(textposition="top center")
    fig.show()

In [13]:
if not gmx_pool.empty:
    latest_pool = gmx_pool.groupby("base")["pool_usd_proxy"].last().reset_index()
    b_avg = (
        binance_vol[binance_vol["base"].isin(UNIVERSE)]
        .groupby("base")[f"vol_ma{MA_WINDOW}d"]
        .mean()
        .reset_index()
    )
    b_avg.columns = ["base", "binance_avg_vol"]

    scatter_pool = latest_pool.merge(b_avg, on="base").dropna()
    fig = px.scatter(
        scatter_pool,
        x="binance_avg_vol",
        y="pool_usd_proxy",
        text="base",
        size="pool_usd_proxy",
        title=f"Binance Avg {MA_WINDOW}d Volume vs GMX Pool Depth (stablecoin USD proxy)",
        labels={
            "binance_avg_vol": "Binance Avg Vol (USD/day)",
            "pool_usd_proxy": "GMX Stable Pool (USD)",
        },
        template="plotly_dark",
        trendline="ols",
        trendline_color_override="orange",
    )
    fig.update_traces(textposition="top center")
    fig.show()

In [14]:
if not gmx_oi.empty:
    print("Pearson correlation: Binance volume MA vs GMX OI (per market)")
    results = []
    for base in UNIVERSE:
        b = binance_vol[binance_vol["base"] == base][["date", f"vol_ma{MA_WINDOW}d"]].set_index(
            "date"
        )
        g = gmx_oi[gmx_oi["base"] == base][["date", "oi_usd"]].set_index("date")
        joined = b.join(g, how="inner").dropna()
        if len(joined) < 30:
            continue
        corr = joined[f"vol_ma{MA_WINDOW}d"].corr(joined["oi_usd"])
        results.append({"base": base, "corr_binance_oi": corr, "n_days": len(joined)})

    corr_df = pd.DataFrame(results).sort_values("corr_binance_oi", ascending=False)
    print(corr_df.to_string(index=False))

    fig = px.bar(
        corr_df,
        x="base",
        y="corr_binance_oi",
        title=f"Pearson Correlation: Binance {MA_WINDOW}d Vol MA vs GMX OI",
        labels={"corr_binance_oi": "Correlation", "base": "Market"},
        template=TEMPLATE,
        color="corr_binance_oi",
        color_continuous_scale="RdYlGn",
        range_color=[-1, 1],
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

Pearson correlation: Binance volume MA vs GMX OI (per market)
    base  corr_binance_oi  n_days
    SHIB         0.795902     503
 VIRTUAL         0.745640     391
     ENA         0.741552     409
    PEPE         0.729380     601
    DOGE         0.712691     786
    NEAR         0.710215     715
     WIF         0.693290     589
     TON         0.678788     465
    BONK         0.657881     439
    WLFI         0.621206     181
   PENGU         0.606165     391
     MEW         0.599912     361
     CRV         0.592590     267
     TIA         0.585247     487
    PUMP         0.580563     230
     POL         0.580150     491
     VVV         0.568133     177
    AVAX         0.563996     735
    BERA         0.557536     378
    AERO         0.551477     194
    BOME         0.540045     400
      OP         0.537367     721
    AAVE         0.530973     742
     STX         0.530486     531
     ARB         0.515558     786
     WLD         0.514646     460
     BNB         0.5

In [15]:
summary_rows = []
for base in sorted(UNIVERSE):
    b_avg_v = (
        binance_vol[binance_vol["base"] == base][f"vol_ma{MA_WINDOW}d"].mean()
        if base in has_binance
        else float("nan")
    )
    h_avg_v = (
        hl_vol[hl_vol["base"] == base][f"vol_ma{MA_WINDOW}d"].mean()
        if base in has_hl
        else float("nan")
    )
    oi_v = (
        gmx_oi[gmx_oi["base"] == base]["oi_usd"].iloc[-1]
        if (not gmx_oi.empty and base in has_gmx_oi)
        else float("nan")
    )
    pool_v = (
        gmx_pool[gmx_pool["base"] == base]["pool_usd_proxy"].iloc[-1]
        if (not gmx_pool.empty and base in has_gmx_pool)
        else float("nan")
    )

    def fmt(v, scale=1e6):
        return round(v / scale, 1) if not np.isnan(v) else "—"

    summary_rows.append(
        {
            "Base": base,
            "Binance Vol $M/d": fmt(b_avg_v),
            "HL Vol $M/d": fmt(h_avg_v),
            "GMX OI $M": fmt(oi_v),
            "GMX Stable Pool $M": fmt(pool_v),
            "On Binance": "✓" if base in has_binance else "—",
            "On HL": "✓" if base in has_hl else "—",
        }
    )

summary_df = pd.DataFrame(summary_rows)
# Sort by Binance volume where available, else HL
summary_df["_sort"] = summary_df["Binance Vol $M/d"].apply(
    lambda x: x if isinstance(x, float) else 0
)
summary_df = summary_df.sort_values("_sort", ascending=False).drop(columns="_sort")

print(f"=== Cross-Exchange Summary — {len(summary_df)} GMX Markets ===")
print(summary_df.to_string(index=False))

=== Cross-Exchange Summary — 97 GMX Markets ===
    Base Binance Vol $M/d HL Vol $M/d GMX OI $M GMX Stable Pool $M On Binance On HL
     BTC          17054.3     15255.4       6.5               38.4          ✓     ✓
     ETH          12125.9      9055.8      10.4               31.6          ✓     ✓
     SOL           3589.8      2100.4       0.9                3.7          ✓     ✓
    DOGE           1441.2      1118.4       0.2                1.4          ✓     ✓
     XRP           1202.9       120.8       2.3                2.6          ✓     ✓
    PEPE           1003.7       882.5       0.0                0.2          ✓     ✓
     SUI            649.7       545.5       0.2                0.5          ✓     ✓
   ASTER            612.0        88.8       0.0                0.1          ✓     ✓
   TRUMP            610.3        11.8       0.0                0.1          ✓     ✓
     BNB            558.2        28.9       0.1                0.4          ✓     ✓
     WIF            481.0   

---
## All GMX Markets — OI & Pool Changes Over Time

Load **every** GMX market and visualise weekly OI and pool liquidity trends. Markets are split into groups of 15 to keep charts readable.

In [16]:
def load_gmx_oi_all(oi_dir: Path) -> pd.DataFrame:
    """Load GMX OI for every market in oi_dir (not filtered by COMMON_BASES).

    :param oi_dir: Root directory containing per-symbol subdirectories.
    :return: DataFrame with columns date, base, oi_usd (daily, summed across market variants).
    """
    frames = []
    for symbol_dir in sorted(oi_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "data.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            # Skip deprecated/test markets
            if any(x in sym_clean for x in ["deprecated", "XAUT_", "CC_", "DOLO_", "KTA_"]):
                continue
            df["oi_usd"] = pd.to_numeric(df["nextValueUsd"], errors="coerce") / SCALE_30
            df["ts"] = pd.to_datetime(df["blockTimestamp"], unit="s", utc=True)
            df["date"] = df["ts"].dt.normalize()
            df["base"] = sym_clean
            daily = df.sort_values("ts").groupby(["date", "base"])["oi_usd"].last().reset_index()
            frames.append(daily)
        except Exception:
            continue
    if not frames:
        return pd.DataFrame(columns=["date", "base", "oi_usd"])
    result = pd.concat(frames, ignore_index=True)
    return result.groupby(["date", "base"])["oi_usd"].sum().reset_index()


def load_gmx_pool_all(pool_dir: Path) -> pd.DataFrame:
    """Load GMX pool liquidity for every market in pool_dir.

    :param pool_dir: Root directory containing per-symbol subdirectories.
    :return: DataFrame with columns date, base, pool_usd_proxy (stablecoin USD proxy).
    """
    frames = []
    for symbol_dir in sorted(pool_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "daily.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            if any(x in sym_clean for x in ["deprecated", "XAUT_", "CC_", "DOLO_", "KTA_"]):
                continue
            df["date"] = pd.to_datetime(df["date"], utc=True)
            df["base"] = sym_clean
            df["token_lc"] = df["token"].str.lower()
            stable = (
                df[df["token_lc"].isin(_STABLE_TOKENS)]
                .groupby(["date", "base"])["pool_tokens"]
                .sum()
                .reset_index()
                .rename(columns={"pool_tokens": "pool_usd_proxy"})
            )
            if not stable.empty:
                frames.append(stable)
        except Exception:
            continue
    if not frames:
        return pd.DataFrame(columns=["date", "base", "pool_usd_proxy"])
    result = pd.concat(frames, ignore_index=True)
    return result.groupby(["date", "base"])["pool_usd_proxy"].sum().reset_index()


print("Loading all GMX OI markets...")
gmx_oi_all = load_gmx_oi_all(GMX_OI_DIR)
print(f"  {gmx_oi_all['base'].nunique()} unique bases, {len(gmx_oi_all)} daily rows")

print("Loading all GMX pool markets...")
gmx_pool_all = load_gmx_pool_all(GMX_POOL_DIR)
print(f"  {gmx_pool_all['base'].nunique()} unique bases, {len(gmx_pool_all)} daily rows")

ALL_GMX_BASES = sorted(gmx_oi_all["base"].unique())
print(f"\nAll GMX bases ({len(ALL_GMX_BASES)}): {ALL_GMX_BASES}")

Loading all GMX OI markets...
  108 unique bases, 41283 daily rows
Loading all GMX pool markets...
  112 unique bases, 45109 daily rows

All GMX bases (108): ['0G', 'AAVE', 'ADA', 'AERO', 'AI16Z', 'AIXBT', 'ALGO', 'ANIME', 'APE', 'APT', 'AR', 'ARB', 'ASTER', 'ATOM', 'AVAX', 'AVNT', 'BCH', 'BERA', 'BNB', 'BOME', 'BONK', 'BRETT', 'BTC', 'CAKE', 'CC', 'CHZ', 'CRO', 'CRV', 'CVX', 'DASH', 'DOGE', 'DOLO', 'DOT', 'DYDX', 'EIGEN', 'ENA', 'ETH', 'FARTCOIN', 'FET', 'FIL', 'FLOKI', 'GMX', 'HBAR', 'HYPE', 'ICP', 'INJ', 'IP', 'JTO', 'JUP', 'KAS', 'KTA', 'LDO', 'LINEA', 'LINK', 'LIT', 'LTC', 'MELANIA', 'MEME', 'MET', 'MEW', 'MKR', 'MNT', 'MON', 'MOODENG', 'MORPHO', 'NEAR', 'OKB', 'OM', 'ONDO', 'OP', 'ORDI', 'PENDLE', 'PENGU', 'PEPE', 'PI', 'POL', 'PUMP', 'RENDER', 'S', 'SATS', 'SEI', 'SHIB', 'SKY', 'SOL', 'SPX6900', 'STX', 'SUI', 'SYRUP', 'TAO', 'TIA', 'TON', 'TRUMP', 'TRX', 'UNI', 'VIRTUAL', 'VVV', 'WELL', 'WIF', 'WLD', 'WLFI', 'XAUT.v2', 'XLM', 'XMR', 'XPL', 'XRP', 'ZEC', 'ZORA', 'ZRO']


In [17]:
# Rank all bases by avg OI so we can order them meaningfully in charts
_oi_rank = gmx_oi_all.groupby("base")["oi_usd"].mean().sort_values(ascending=False).reset_index()
_oi_rank.columns = ["base", "avg_oi_usd"]
print("Top 20 GMX markets by avg OI (USD M):")
print(
    (
        _oi_rank.head(20).assign(avg_oi_M=lambda d: (d["avg_oi_usd"] / 1e6).round(1))[
            ["base", "avg_oi_M"]
        ]
    ).to_string(index=False)
)


# Resample weekly to smooth out noise for multi-market charts
def resample_weekly(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Resample daily data to weekly using last value.

    :param df: DataFrame with date, base, and value column.
    :param value_col: Column name to aggregate.
    :return: Weekly DataFrame with same columns.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["base", "date"])
    return df.set_index("date").groupby("base")[value_col].resample("W").last().reset_index()


gmx_oi_weekly = resample_weekly(gmx_oi_all, "oi_usd")
gmx_pool_weekly = resample_weekly(gmx_pool_all, "pool_usd_proxy")
print(f"\nWeekly OI: {len(gmx_oi_weekly)} rows | Weekly Pool: {len(gmx_pool_weekly)} rows")

Top 20 GMX markets by avg OI (USD M):
    base  avg_oi_M
     BTC      21.8
     ETH      18.9
     SOL       4.4
    LINK       3.0
     XRP       0.9
    DOGE       0.9
     ARB       0.9
     SUI       0.8
     GMX       0.6
    HYPE       0.6
    AVAX       0.4
    AAVE       0.3
FARTCOIN       0.3
     CRV       0.3
     ENA       0.3
    NEAR       0.3
    PEPE       0.2
 XAUT.v2       0.2
     XMR       0.2
     XPL       0.2

Weekly OI: 6307 rows | Weekly Pool: 6773 rows


In [18]:
def plot_oi_group(bases: list[str], weekly_df: pd.DataFrame, title_suffix: str) -> None:
    """Plot weekly OI over time for a group of markets on a single chart.

    :param bases: List of base symbols to include.
    :param weekly_df: Weekly OI DataFrame with columns date, base, oi_usd.
    :param title_suffix: Appended to chart title to identify the group.
    """
    df = weekly_df[weekly_df["base"].isin(bases)]
    fig = px.line(
        df,
        x="date",
        y="oi_usd",
        color="base",
        title=f"GMX Open Interest Over Time — {title_suffix}",
        labels={"oi_usd": "OI (USD)", "date": "Date", "base": "Market"},
        template=TEMPLATE,
    )
    fig.update_layout(
        hovermode="x unified",
        yaxis_tickformat="$,.0f",
        legend=dict(orientation="v", x=1.01, y=1),
        height=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.show()


# Split ALL_GMX_BASES by OI rank (ordered highest → lowest), groups of 15
GROUP_SIZE = 15
ranked_bases = _oi_rank["base"].tolist()
groups = [ranked_bases[i : i + GROUP_SIZE] for i in range(0, len(ranked_bases), GROUP_SIZE)]

for idx, group in enumerate(groups):
    label = f"Group {idx + 1}/{len(groups)} (rank {idx * GROUP_SIZE + 1}–{min((idx + 1) * GROUP_SIZE, len(ranked_bases))} by OI)"
    plot_oi_group(group, gmx_oi_weekly, label)

In [19]:
def plot_pool_group(bases: list[str], weekly_df: pd.DataFrame, title_suffix: str) -> None:
    """Plot weekly pool liquidity (stablecoin USD proxy) over time for a group of markets.

    :param bases: List of base symbols to include.
    :param weekly_df: Weekly pool DataFrame with columns date, base, pool_usd_proxy.
    :param title_suffix: Appended to chart title to identify the group.
    """
    df = weekly_df[weekly_df["base"].isin(bases)]
    if df.empty:
        print(f"  No pool data for {title_suffix}")
        return
    fig = px.line(
        df,
        x="date",
        y="pool_usd_proxy",
        color="base",
        title=f"GMX Pool Depth (Stablecoin USD Proxy) — {title_suffix}",
        labels={"pool_usd_proxy": "Stable Pool (USD)", "date": "Date", "base": "Market"},
        template=TEMPLATE,
    )
    fig.update_layout(
        hovermode="x unified",
        yaxis_tickformat="$,.0f",
        legend=dict(orientation="v", x=1.01, y=1),
        height=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.show()


pool_ranked_bases = (
    gmx_pool_all.groupby("base")["pool_usd_proxy"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
pool_groups = [
    pool_ranked_bases[i : i + GROUP_SIZE] for i in range(0, len(pool_ranked_bases), GROUP_SIZE)
]

for idx, group in enumerate(pool_groups):
    label = f"Group {idx + 1}/{len(pool_groups)} (rank {idx * GROUP_SIZE + 1}–{min((idx + 1) * GROUP_SIZE, len(pool_ranked_bases))} by pool depth)"
    plot_pool_group(group, gmx_pool_weekly, label)

In [20]:
# OI utilisation heatmap: monthly avg OI per market (top 30)
_top30_bases = _oi_rank.head(30)["base"].tolist()
_monthly_oi = (
    gmx_oi_all[gmx_oi_all["base"].isin(_top30_bases)]
    .assign(month=lambda d: pd.to_datetime(d["date"]).dt.to_period("M"))
    .groupby(["month", "base"])["oi_usd"]
    .mean()
    .reset_index()
)
_monthly_oi["month_str"] = _monthly_oi["month"].astype(str)
_pivot = (
    _monthly_oi.pivot(index="base", columns="month_str", values="oi_usd")
    .fillna(0)
    .reindex(_top30_bases)
)

fig = px.imshow(
    _pivot / 1e6,
    title="GMX Monthly Avg OI per Market — Top 30 (USD millions)",
    labels={"color": "OI $M", "x": "Month", "y": "Market"},
    color_continuous_scale="YlOrRd",
    template=TEMPLATE,
    aspect="auto",
)
fig.update_layout(
    height=700,
    paper_bgcolor="white",
    xaxis_tickangle=-45,
)
fig.show()

---
## GMX Trading Universe Filter

Select the best pairs for GMX trading using a three-stage funnel:

1. **Binance 30d rolling volume MA** — high CEX volume = liquid, price-discovered market  
2. **GMX pool depth (stablecoin USD proxy)** — enough USDC in the pool to enter/exit  
3. **GMX open interest** — active market with real traders  
4. **Longevity filter** — minimum 2 years of GMX on-chain data (mature market)

Markets are ranked on each signal separately, then combined into a composite score (lower = better).

In [21]:
# ── Step 1: Compute how many days each GMX base has OI data ─────────────────
MIN_YEARS = 2
MIN_DAYS = int(MIN_YEARS * 365.25)

gmx_longevity = (
    gmx_oi_all.assign(date=lambda d: pd.to_datetime(d["date"]))
    .groupby("base")["date"]
    .agg(first_date="min", last_date="max")
    .reset_index()
)
gmx_longevity["days_of_data"] = (gmx_longevity["last_date"] - gmx_longevity["first_date"]).dt.days
gmx_longevity["has_min_history"] = gmx_longevity["days_of_data"] >= MIN_DAYS

mature_bases = set(gmx_longevity.loc[gmx_longevity["has_min_history"], "base"])
print(f"GMX bases with {MIN_YEARS}+ years of data ({MIN_DAYS}+ days): {len(mature_bases)}")
print(sorted(mature_bases))

short_history = sorted(set(gmx_oi_all["base"].unique()) - mature_bases)
print(f"\nExcluded (< {MIN_YEARS} yrs): {short_history}")

GMX bases with 2+ years of data (730+ days): 14
['AAVE', 'ARB', 'ATOM', 'AVAX', 'BNB', 'BTC', 'DOGE', 'ETH', 'LINK', 'LTC', 'NEAR', 'SOL', 'UNI', 'XRP']

Excluded (< 2 yrs): ['0G', 'ADA', 'AERO', 'AI16Z', 'AIXBT', 'ALGO', 'ANIME', 'APE', 'APT', 'AR', 'ASTER', 'AVNT', 'BCH', 'BERA', 'BOME', 'BONK', 'BRETT', 'CAKE', 'CC', 'CHZ', 'CRO', 'CRV', 'CVX', 'DASH', 'DOLO', 'DOT', 'DYDX', 'EIGEN', 'ENA', 'FARTCOIN', 'FET', 'FIL', 'FLOKI', 'GMX', 'HBAR', 'HYPE', 'ICP', 'INJ', 'IP', 'JTO', 'JUP', 'KAS', 'KTA', 'LDO', 'LINEA', 'LIT', 'MELANIA', 'MEME', 'MET', 'MEW', 'MKR', 'MNT', 'MON', 'MOODENG', 'MORPHO', 'OKB', 'OM', 'ONDO', 'OP', 'ORDI', 'PENDLE', 'PENGU', 'PEPE', 'PI', 'POL', 'PUMP', 'RENDER', 'S', 'SATS', 'SEI', 'SHIB', 'SKY', 'SPX6900', 'STX', 'SUI', 'SYRUP', 'TAO', 'TIA', 'TON', 'TRUMP', 'TRX', 'VIRTUAL', 'VVV', 'WELL', 'WIF', 'WLD', 'WLFI', 'XAUT.v2', 'XLM', 'XMR', 'XPL', 'ZEC', 'ZORA', 'ZRO']


In [22]:
# ── Step 2: Build composite score and ranked universe ────────────────────────
# Best available CEX volume: prefer Binance, fall back to Hyperliquid
_bin_ma = (
    binance_vol.sort_values(["base", "date"])
    .groupby("base")
    .tail(MA_WINDOW)
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .rename("binance_ma")
)
_hl_ma = (
    hl_vol.sort_values(["base", "date"])
    .groupby("base")
    .tail(MA_WINDOW)
    .groupby("base")[f"vol_ma{MA_WINDOW}d"]
    .mean()
    .rename("hl_ma")
)
_cex_combined = pd.concat([_bin_ma, _hl_ma], axis=1)
_cex_combined["cex_vol_ma"] = _cex_combined["binance_ma"].combine_first(_cex_combined["hl_ma"])
cex_latest_ma = _cex_combined.reset_index()[["base", "binance_ma", "hl_ma", "cex_vol_ma"]]

# Latest GMX pool depth
pool_latest = (
    gmx_pool_all.sort_values(["base", "date"])
    .groupby("base")["pool_usd_proxy"]
    .last()
    .reset_index(name="pool_latest")
)

# Latest GMX OI
oi_latest = (
    gmx_oi_all.sort_values(["base", "date"])
    .groupby("base")["oi_usd"]
    .last()
    .reset_index(name="oi_latest")
)

# Merge: start from all GMX bases that have longevity data
universe_raw = (
    gmx_longevity[["base", "days_of_data", "has_min_history"]]
    .merge(cex_latest_ma, on="base", how="left")
    .merge(pool_latest, on="base", how="left")
    .merge(oi_latest, on="base", how="left")
)

# ── Longevity filter ──────────────────────────────────────────────────────────
universe_filtered = universe_raw[universe_raw["has_min_history"] == True].copy()

# ── Individual signal ranks (1 = best, NaN → last) ───────────────────────────
universe_filtered["rank_volume"] = universe_filtered["cex_vol_ma"].rank(
    ascending=False, method="min", na_option="bottom"
)
universe_filtered["rank_pool"] = universe_filtered["pool_latest"].rank(
    ascending=False, method="min", na_option="bottom"
)
universe_filtered["rank_oi"] = universe_filtered["oi_latest"].rank(
    ascending=False, method="min", na_option="bottom"
)

# ── Composite score: equal-weight average rank ────────────────────────────────
universe_filtered["composite_score"] = (
    universe_filtered["rank_volume"] + universe_filtered["rank_pool"] + universe_filtered["rank_oi"]
) / 3.0

universe_filtered = universe_filtered.sort_values("composite_score").reset_index(drop=True)
universe_filtered["rank_final"] = range(1, len(universe_filtered) + 1)

# ── Display ───────────────────────────────────────────────────────────────────
display_df = universe_filtered[
    [
        "rank_final",
        "base",
        "cex_vol_ma",
        "binance_ma",
        "hl_ma",
        "pool_latest",
        "oi_latest",
        "days_of_data",
        "composite_score",
    ]
].copy()
display_df.columns = [
    "#",
    "Base",
    "CEX Vol MA $M/d",
    "└ Binance",
    "└ HL",
    "Pool $M",
    "OI $M",
    "GMX Days",
    "Score",
]
for col in ["CEX Vol MA $M/d", "└ Binance", "└ HL"]:
    display_df[col] = (display_df[col] / 1e6).round(1)
display_df["Pool $M"] = (display_df["Pool $M"] / 1e6).round(2)
display_df["OI $M"] = (display_df["OI $M"] / 1e6).round(2)
display_df["Score"] = display_df["Score"].round(2)

print(f"=== GMX Trading Universe — Composite Rank ({MIN_YEARS}+ years filter) ===")
print(f"Ranked by: CEX Volume MA + Pool Depth + OI  (lower score = better)")
print(f"CEX volume = Binance if available, else Hyperliquid")
print()
print(display_df.to_string(index=False))

RANKED_UNIVERSE = universe_filtered["base"].tolist()
print(f"\nTop-10 recommendation: {RANKED_UNIVERSE[:10]}")

=== GMX Trading Universe — Composite Rank (2+ years filter) ===
Ranked by: CEX Volume MA + Pool Depth + OI  (lower score = better)
CEX volume = Binance if available, else Hyperliquid

 # Base  CEX Vol MA $M/d  └ Binance    └ HL  Pool $M  OI $M  GMX Days  Score
 1  BTC          15536.0    15536.0 14427.5    38.38   6.52       943   1.33
 2  ETH          13464.4    13464.4 13452.4    31.64  10.38       943   1.67
 3  SOL           2854.5     2854.5  2971.7     3.74   0.88       943   3.67
 4  XRP           1364.7     1364.7    99.0     2.58   2.31       943   4.00
 5 LINK            176.7      176.7   190.2     3.19   1.37       943   5.00
 6 DOGE            650.6      650.6   660.7     1.36   0.20       943   6.00
 7  BNB            566.1      566.1    10.0     0.41   0.10       776   7.00
 8 AVAX            172.6      172.6   177.0     0.12   0.22       734   9.00
 9  LTC            133.3      133.3     6.5     0.13   0.02       943  10.00
10  UNI            107.3      107.3   101.7   

In [ ]:
# ── Step 3: Scatter — Binance Volume vs GMX Pool Depth ───────────────────────
# Bubble size = OI; colour = composite score (green = best rank)
scatter_data = universe_filtered.dropna(subset=["binance_ma", "pool_latest", "oi_latest"]).copy()
scatter_data["oi_latest_safe"] = scatter_data["oi_latest"].clip(lower=1e4)  # avoid zero bubbles

fig = px.scatter(
    scatter_data,
    x="binance_ma",
    y="pool_latest",
    size="oi_latest_safe",
    text="base",
    color="composite_score",
    color_continuous_scale="RdYlGn_r",  # green = low score = best
    title=f"Binance Volume vs GMX Pool Depth<br>"
    f"<sup>Bubble size = OI  |  Green = top composite rank  |  Filter: {MIN_YEARS}+ yrs GMX data</sup>",
    labels={
        "binance_ma": f"Binance {MA_WINDOW}d Vol MA (USD/day)",
        "pool_latest": "GMX Pool Depth — Stablecoin USD proxy",
        "oi_latest_safe": "GMX OI (USD)",
        "composite_score": "Composite\nScore",
    },
    template=TEMPLATE,
    size_max=60,
)
fig.update_traces(textposition="top center")
fig.update_layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=600,
)
fig.show()

# Bar chart: per-signal ranks side by side
rank_melt = universe_filtered.melt(
    id_vars="base",
    value_vars=["rank_volume", "rank_pool", "rank_oi"],
    var_name="signal",
    value_name="rank",
)
rank_melt["signal"] = rank_melt["signal"].map(
    {
        "rank_volume": "Binance Volume",
        "rank_pool": "GMX Pool Depth",
        "rank_oi": "GMX OI",
    }
)
# Order x-axis by composite rank
rank_melt["base"] = pd.Categorical(rank_melt["base"], categories=RANKED_UNIVERSE, ordered=True)

fig2 = px.bar(
    rank_melt.sort_values("base"),
    x="base",
    y="rank",
    color="signal",
    barmode="group",
    title="Per-Signal Ranks (lower bar = better)",
    labels={"rank": "Rank (1 = best)", "base": "Market", "signal": "Signal"},
    template=TEMPLATE,
    color_discrete_map={
        "Binance Volume": "#1f77b4",
        "GMX Pool Depth": "#9467bd",
        "GMX OI": "#2ca02c",
    },
)
fig2.update_layout(
    xaxis_tickangle=-45,
    paper_bgcolor="white",
    plot_bgcolor="white",
    yaxis=dict(autorange="reversed"),  # rank 1 at top
)
fig2.show()

In [ ]:
# ── Step 4: Binance Volume vs GMX Liquidity Flow — top-ranked pairs ──────────
# Shows how CEX volume and GMX pool depth co-move over time
TOP_N = min(5, len(RANKED_UNIVERSE))
top_bases = RANKED_UNIVERSE[:TOP_N]
print(f"Plotting liquidity flow for top {TOP_N}: {top_bases}")

for base in top_bases:
    b_df = binance_vol[binance_vol["base"] == base].sort_values("date")
    pool_df = gmx_pool_all[gmx_pool_all["base"] == base].sort_values("date")
    oi_df = gmx_oi_all[gmx_oi_all["base"] == base].sort_values("date")

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=[
            f"Binance Volume (USD/day)  [{MA_WINDOW}d MA]",
            "GMX Pool Depth — Stablecoin USD proxy ($M)",
            "GMX Open Interest ($M)",
        ],
        vertical_spacing=0.07,
    )

    # Row 1 — Binance volume bars + MA
    fig.add_trace(
        go.Bar(
            x=b_df["date"],
            y=b_df["usd_volume"],
            name="Binance Vol (raw)",
            marker_color="#1f77b4",
            opacity=0.25,
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=b_df["date"],
            y=b_df[f"vol_ma{MA_WINDOW}d"],
            name=f"Binance {MA_WINDOW}d MA",
            line=dict(color="#1f77b4", width=2),
        ),
        row=1,
        col=1,
    )

    # Row 2 — GMX pool depth (stablecoin leg in $M)
    if not pool_df.empty:
        fig.add_trace(
            go.Scatter(
                x=pool_df["date"],
                y=pool_df["pool_usd_proxy"] / 1e6,
                name="GMX Pool ($M USDC)",
                line=dict(color="#9467bd", width=2),
                fill="tozeroy",
                fillcolor="rgba(148,103,189,0.12)",
            ),
            row=2,
            col=1,
        )
    else:
        fig.add_annotation(text="No pool data", row=2, col=1, showarrow=False)

    # Row 3 — GMX OI in $M
    if not oi_df.empty:
        fig.add_trace(
            go.Scatter(
                x=oi_df["date"],
                y=oi_df["oi_usd"] / 1e6,
                name="GMX OI ($M)",
                line=dict(color="#2ca02c", width=2),
                fill="tozeroy",
                fillcolor="rgba(44,160,44,0.12)",
            ),
            row=3,
            col=1,
        )
    else:
        fig.add_annotation(text="No OI data", row=3, col=1, showarrow=False)

    # Score annotation
    row_score = universe_filtered[universe_filtered["base"] == base]
    score_text = (
        f"Composite score: {row_score['composite_score'].iloc[0]:.2f}  "
        f"| Vol rank: {int(row_score['rank_volume'].iloc[0])}  "
        f"| Pool rank: {int(row_score['rank_pool'].iloc[0])}  "
        f"| OI rank: {int(row_score['rank_oi'].iloc[0])}"
        if not row_score.empty
        else ""
    )

    fig.update_layout(
        title=f"{base}/USD — Binance Volume vs GMX Liquidity Flow<br><sup>{score_text}</sup>",
        template=TEMPLATE,
        hovermode="x unified",
        height=750,
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="h", y=-0.05),
    )
    fig.update_yaxes(title_text="USD/day", row=1, col=1, tickformat="$,.0f")
    fig.update_yaxes(title_text="Pool ($M)", row=2, col=1, tickformat="$,.1f")
    fig.update_yaxes(title_text="OI ($M)", row=3, col=1, tickformat="$,.1f")
    fig.show()

Plotting liquidity flow for top 5: ['BTC', 'ETH', 'SOL', 'LINK', 'DOGE']


---
## Newer GMX Markets — Less Than 2 Years of Data

These markets are listed on Binance but have fewer than 2 years of GMX on-chain history.  
They may still be worth trading but carry **higher model uncertainty** (less historical context for strategy calibration).

Ranked by the same composite score (Binance volume + pool depth + OI) within this sub-universe.

In [ ]:
# ── Newer markets: Binance-listed but < 2 years on GMX ───────────────────────
newer_bases_all = sorted(set(_binance_bases) - mature_bases)
print(f"Binance-listed bases with < {MIN_YEARS} yrs GMX data: {newer_bases_all}")

# Build the same composite table for these bases
newer_raw = (
    cex_latest_ma[cex_latest_ma["base"].isin(newer_bases_all)]
    .merge(pool_latest, on="base", how="left")
    .merge(oi_latest, on="base", how="left")
    .merge(
        gmx_longevity[["base", "days_of_data", "first_date", "last_date"]], on="base", how="left"
    )
)

# Some Binance bases may have zero GMX data at all → fill days with 0
newer_raw["days_of_data"] = newer_raw["days_of_data"].fillna(0).astype(int)
newer_raw["first_date"] = newer_raw["first_date"].fillna(pd.NaT)

# Rank within this sub-universe
newer_raw["rank_volume"] = newer_raw["binance_ma"].rank(ascending=False, method="min")
newer_raw["rank_pool"] = newer_raw["pool_latest"].rank(
    ascending=False, method="min", na_option="bottom"
)
newer_raw["rank_oi"] = newer_raw["oi_latest"].rank(
    ascending=False, method="min", na_option="bottom"
)
newer_raw["composite_score"] = (
    newer_raw["rank_volume"] + newer_raw["rank_pool"] + newer_raw["rank_oi"]
) / 3.0
newer_raw = newer_raw.sort_values("composite_score").reset_index(drop=True)
newer_raw["rank_final"] = range(1, len(newer_raw) + 1)

NEWER_RANKED = newer_raw["base"].tolist()

# ── Display table ─────────────────────────────────────────────────────────────
disp = newer_raw[
    [
        "rank_final",
        "base",
        "binance_ma",
        "pool_latest",
        "oi_latest",
        "days_of_data",
        "first_date",
    ]
].copy()
disp.columns = [
    "#",
    "Base",
    "Binance Vol MA ($M/d)",
    "Pool Depth ($M)",
    "OI ($M)",
    "GMX Days",
    "GMX Since",
]
disp["Binance Vol MA ($M/d)"] = (disp["Binance Vol MA ($M/d)"] / 1e6).round(1)
disp["Pool Depth ($M)"] = (disp["Pool Depth ($M)"] / 1e6).round(2)
disp["OI ($M)"] = (disp["OI ($M)"] / 1e6).round(2)
disp["GMX Since"] = pd.to_datetime(disp["GMX Since"]).dt.strftime("%Y-%m-%d").fillna("not on GMX")

print(f"\n=== Newer GMX Markets (< {MIN_YEARS} yrs) — Composite Rank ===\n")
print(disp.to_string(index=False))

In [ ]:
# ── Scatter: newer markets — Volume vs Pool (same layout as mature section) ───
scatter_newer = newer_raw.dropna(subset=["binance_ma"]).copy()
scatter_newer["oi_safe"] = scatter_newer["oi_latest"].fillna(0).clip(lower=1e4)
scatter_newer["pool_safe"] = scatter_newer["pool_latest"].fillna(0)
scatter_newer["days_label"] = scatter_newer["days_of_data"].apply(
    lambda d: f"{d}d" if d > 0 else "not on GMX"
)

fig = px.scatter(
    scatter_newer,
    x="binance_ma",
    y="pool_safe",
    size="oi_safe",
    text="base",
    color="composite_score",
    color_continuous_scale="RdYlGn_r",
    hover_data={"days_label": True, "pool_safe": False, "oi_safe": False},
    title=f"Newer Markets — Binance Volume vs GMX Pool Depth<br>"
    f"<sup>Bubble size = OI  |  Green = better composite rank  |  Filter: &lt; {MIN_YEARS} yrs GMX data</sup>",
    labels={
        "binance_ma": f"Binance {MA_WINDOW}d Vol MA (USD/day)",
        "pool_safe": "GMX Pool Depth — Stablecoin USD proxy",
        "oi_safe": "GMX OI (USD)",
        "composite_score": "Score",
        "days_label": "GMX History",
    },
    template=TEMPLATE,
    size_max=60,
)
fig.update_traces(textposition="top center")
fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", height=550)
fig.show()

# ── Bar: days of GMX history per newer base ───────────────────────────────────
fig2 = px.bar(
    scatter_newer.sort_values("days_of_data", ascending=False),
    x="base",
    y="days_of_data",
    color="days_of_data",
    color_continuous_scale="Blues",
    title=f"GMX Data History — Newer Markets (all < {MIN_DAYS} days)",
    labels={"days_of_data": "Days on GMX", "base": "Market"},
    template=TEMPLATE,
    text="days_label",
)
fig2.add_hline(
    y=MIN_DAYS,
    line_dash="dash",
    line_color="red",
    annotation_text=f"{MIN_YEARS}-year threshold ({MIN_DAYS}d)",
    annotation_position="top right",
)
fig2.update_traces(textposition="outside")
fig2.update_layout(
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
    xaxis_tickangle=-45,
)
fig2.show()

In [ ]:
# ── Liquidity flow for top newer markets (those that ARE on GMX) ──────────────
newer_on_gmx = [b for b in NEWER_RANKED if b in set(gmx_oi_all["base"].unique())]
TOP_NEWER = min(5, len(newer_on_gmx))
print(f"Plotting liquidity flow for top {TOP_NEWER} newer bases: {newer_on_gmx[:TOP_NEWER]}")

for base in newer_on_gmx[:TOP_NEWER]:
    b_df = binance_vol[binance_vol["base"] == base].sort_values("date")
    pool_df = gmx_pool_all[gmx_pool_all["base"] == base].sort_values("date")
    oi_df = gmx_oi_all[gmx_oi_all["base"] == base].sort_values("date")

    row_info = newer_raw[newer_raw["base"] == base]
    days = int(row_info["days_of_data"].iloc[0]) if not row_info.empty else 0
    since = (
        pd.to_datetime(row_info["first_date"].iloc[0]).strftime("%Y-%m-%d")
        if not row_info.empty and pd.notna(row_info["first_date"].iloc[0])
        else "unknown"
    )
    score_text = (
        f"Score: {row_info['composite_score'].iloc[0]:.2f}  "
        f"| GMX since {since} ({days}d)  "
        f"| Vol rank: {int(row_info['rank_volume'].iloc[0])}  "
        f"| Pool rank: {int(row_info['rank_pool'].iloc[0])}  "
        f"| OI rank: {int(row_info['rank_oi'].iloc[0])}"
        if not row_info.empty
        else ""
    )

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=[
            f"Binance Volume (USD/day) [{MA_WINDOW}d MA]",
            "GMX Pool Depth — Stablecoin USD proxy ($M)",
            "GMX Open Interest ($M)",
        ],
        vertical_spacing=0.07,
    )
    fig.add_trace(
        go.Bar(
            x=b_df["date"],
            y=b_df["usd_volume"],
            name="Binance Vol",
            marker_color="#1f77b4",
            opacity=0.25,
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=b_df["date"],
            y=b_df[f"vol_ma{MA_WINDOW}d"],
            name=f"{MA_WINDOW}d MA",
            line=dict(color="#1f77b4", width=2),
        ),
        row=1,
        col=1,
    )
    if not pool_df.empty:
        fig.add_trace(
            go.Scatter(
                x=pool_df["date"],
                y=pool_df["pool_usd_proxy"] / 1e6,
                name="GMX Pool ($M)",
                line=dict(color="#9467bd", width=2),
                fill="tozeroy",
                fillcolor="rgba(148,103,189,0.12)",
            ),
            row=2,
            col=1,
        )
    if not oi_df.empty:
        fig.add_trace(
            go.Scatter(
                x=oi_df["date"],
                y=oi_df["oi_usd"] / 1e6,
                name="GMX OI ($M)",
                line=dict(color="#2ca02c", width=2),
                fill="tozeroy",
                fillcolor="rgba(44,160,44,0.12)",
            ),
            row=3,
            col=1,
        )
    fig.update_layout(
        title=f"{base}/USD — Newer Market ({days}d GMX history)<br><sup>{score_text}</sup>",
        template=TEMPLATE,
        hovermode="x unified",
        height=750,
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="h", y=-0.05),
    )
    fig.update_yaxes(title_text="USD/day", row=1, col=1, tickformat="$,.0f")
    fig.update_yaxes(title_text="Pool ($M)", row=2, col=1, tickformat="$,.1f")
    fig.update_yaxes(title_text="OI ($M)", row=3, col=1, tickformat="$,.1f")
    fig.show()

Plotting liquidity flow for top 5 newer bases: ['XMR', 'SUI', 'ENA', 'PENDLE', 'WIF']


---
## Full GMX Universe — All Markets

True pool USD using DeFiLlama token prices (monthly snapshots, 32 HTTP calls total).  
Covers every GMX market — not limited to the Binance/Hyperliquid intersection.

In [ ]:
import calendar
import time
import urllib.request
import json as _json

# ── Token address → CoinGecko ID mapping ────────────────────────────────────
# Stablecoins: price = $1 (no lookup needed)
_STABLE_TOKENS_LC = {
    "0xaf88d065e77c8cc2239327c5edb3a432268e5831",  # USDC native
    "0xff970a61a04b1ca14834a43f5de4533ebddb5cc8",  # USDC.e
    "0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9",  # USDT
    "0xda10009cbd5d07dd0cecc66161fc93d7c9000da1",  # DAI
    "0x5d3a1ff2b6bab83b63cd9ad0787074081a52ef34",  # USDe (Ethena)
}

# Non-stable tokens: arbitrum address → coingecko ID
_ADDR_TO_CG = {
    "0x82af49447d8a07e3bd95bd0d56f35241523fbab1": "coingecko:ethereum",  # WETH
    "0x2f2a2543b76a4166549f7aab2e75bef0aefc5b0f": "coingecko:wrapped-bitcoin",  # WBTC
    "0x6c84a8f1c29108f47a79964b5fe888d4f4d0de40": "coingecko:tbtc",  # tBTC
    "0x912ce59144191c1204e64559fe8253a0e49e6548": "coingecko:arbitrum",  # ARB
    "0xf97f4df75117a78c1a5a0dbb814af92458539fb4": "coingecko:chainlink",  # LINK
    "0xba5ddd1f9d7f570dc94a51479a000e3bce967196": "coingecko:aave",  # AAVE
    "0xfa7f8980b0f1e64a2062791cc3b0871572f1f7f0": "coingecko:uniswap",  # UNI
    "0xfc5a1a6eb076a2c7ad06ed22c90d7e710e35ad0a": "coingecko:gmx",  # GMX
    "0x2bcc6d6cdbbdc0a4071e48bb3b969b06b3330c07": "coingecko:wrapped-solana",  # wSOL
    "0x5979d7b546e38e414f7e9822514be443a4800529": "coingecko:wrapped-steth",  # wstETH
    "0x565609faf65b92f7be02468acf86f8979423e514": "coingecko:avalanche-2",  # WAVAX
    "0xa9004a5421372e1d83fb1f85b0fc986c912f91f3": "coingecko:wrapped-bnb",  # WBNB
    "0x0c880f6761f1af8d9aa9c466984b80dab9a8c9e8": "coingecko:pendle",  # PENDLE
    "0x25d887ce7a35172c62febfd67a1856f20faebb00": "coingecko:pepe",  # PEPE
    "0x7f9fbf9bdd3f4105c478b996b648fe6e828a1e98": "coingecko:apecoin",  # APE
    "0xac800fd6159c2a2cb8fc31ef74621eb430287a5a": "coingecko:optimism",  # OP
    "0xa1b91fe9fd52141ff8cac388ce3f10bfdc1de79d": "coingecko:dogwifcoin",  # WIF
    "0x37a645648df29205c6261289983fb04ecd70b4b3": "coingecko:anime",  # ANIME
}


# ── Fetch monthly prices from DeFiLlama (32 requests total) ─────────────────
def fetch_monthly_prices(
    addr_to_cg: dict,
    start_year: int = 2023,
    start_month: int = 8,
    end_year: int = 2026,
    end_month: int = 3,
) -> pd.DataFrame:
    """Fetch monthly close prices for non-stable pool tokens from DeFiLlama.

    :param addr_to_cg: Mapping of token address → coingecko ID.
    :param start_year: First year to fetch.
    :param start_month: First month to fetch.
    :param end_year: Last year to fetch.
    :param end_month: Last month to fetch (inclusive).
    :return: DataFrame with columns token_addr, year_month, price_usd.
    """
    cg_ids = list(set(addr_to_cg.values()))
    coins_param = ",".join(cg_ids)

    rows = []
    total = 0
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            if year == start_year and month < start_month:
                continue
            if year == end_year and month > end_month:
                break
            ts = int(calendar.timegm((year, month, 15, 0, 0, 0, 0, 0, 0)))  # mid-month
            url = f"https://coins.llama.fi/prices/historical/{ts}/{coins_param}"
            try:
                with urllib.request.urlopen(url, timeout=15) as r:
                    data = _json.loads(r.read())
                for cg_id, info in data.get("coins", {}).items():
                    rows.append(
                        {
                            "cg_id": cg_id,
                            "year_month": f"{year:04d}-{month:02d}",
                            "price_usd": info["price"],
                        }
                    )
                total += 1
            except Exception as e:
                print(f"  WARN {year}-{month:02d}: {e}")
            time.sleep(0.05)  # gentle rate limiting

    print(f"Fetched {total} monthly snapshots, {len(rows)} (cg_id, month) price points")
    df = pd.DataFrame(rows)
    # Build reverse map: cg_id → [token_addrs]
    cg_to_addrs = {}
    for addr, cg in addr_to_cg.items():
        cg_to_addrs.setdefault(cg, []).append(addr)
    # Expand: one row per token address
    expanded = []
    for _, row in df.iterrows():
        for addr in cg_to_addrs.get(row["cg_id"], []):
            expanded.append(
                {"token_addr": addr, "year_month": row["year_month"], "price_usd": row["price_usd"]}
            )
    return pd.DataFrame(expanded)


print("Fetching monthly token prices from DeFiLlama...")
price_monthly = fetch_monthly_prices(_ADDR_TO_CG)
print(f"Price table: {len(price_monthly)} rows, {price_monthly['token_addr'].nunique()} tokens")
print(price_monthly.head(5))

Fetching monthly token prices from DeFiLlama...
Fetched 32 monthly snapshots, 505 (cg_id, month) price points
Price table: 505 rows, 17 tokens
                                   token_addr year_month  price_usd
0  0x2f2a2543b76a4166549f7aab2e75bef0aefc5b0f    2023-08   29442.00
1  0x6c84a8f1c29108f47a79964b5fe888d4f4d0de40    2023-08   28736.00
2  0x5979d7b546e38e414f7e9822514be443a4800529    2023-08    2094.23
3  0x912ce59144191c1204e64559fe8253a0e49e6548    2023-08       1.14
4  0xac800fd6159c2a2cb8fc31ef74621eb430287a5a    2023-08       1.53


In [ ]:
# ── Build true pool USD for ALL GMX markets ──────────────────────────────────
def load_gmx_pool_usd(pool_dir: Path, price_monthly: pd.DataFrame) -> pd.DataFrame:
    """Load pool data for all markets and convert native tokens to USD via DeFiLlama prices.

    :param pool_dir: Root directory with per-symbol ``daily.parquet`` files.
    :param price_monthly: DataFrame with token_addr, year_month, price_usd columns.
    :return: DataFrame with columns date, base, pool_usd_total, pool_usd_stable, pool_usd_native.
    """
    frames = []
    pm = price_monthly.copy()
    pm["token_addr"] = pm["token_addr"].str.lower()

    for symbol_dir in sorted(pool_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        f = symbol_dir / "daily.parquet"
        if not f.exists():
            continue
        try:
            df = pd.read_parquet(f)
            if df.empty:
                continue
            sym = str(df["symbol"].iloc[0]).split("/")[0].strip()
            if any(x in sym for x in ["deprecated", "XAUT_", "CC_", "DOLO_", "KTA_"]):
                continue
            df["date"] = pd.to_datetime(df["date"], utc=True)
            df["base"] = sym
            df["token_lc"] = df["token"].str.lower()
            df["year_month"] = df["date"].dt.strftime("%Y-%m")

            # Stable leg: price = $1
            stable = df[df["token_lc"].isin(_STABLE_TOKENS_LC)].copy()
            stable["pool_usd"] = stable["pool_tokens"]  # 1:1

            # Native leg: join with monthly prices
            native = df[~df["token_lc"].isin(_STABLE_TOKENS_LC)].copy()
            native = native.merge(
                pm,
                left_on=["token_lc", "year_month"],
                right_on=["token_addr", "year_month"],
                how="left",
            )
            native["pool_usd"] = native["pool_tokens"] * native["price_usd"].fillna(0)

            # Aggregate per (date, base)
            daily_stable = (
                stable.groupby(["date", "base"])["pool_usd"]
                .sum()
                .reset_index()
                .rename(columns={"pool_usd": "pool_usd_stable"})
            )
            daily_native = (
                native.groupby(["date", "base"])["pool_usd"]
                .sum()
                .reset_index()
                .rename(columns={"pool_usd": "pool_usd_native"})
            )
            daily = daily_stable.merge(daily_native, on=["date", "base"], how="outer").fillna(0)
            daily["pool_usd_total"] = daily["pool_usd_stable"] + daily["pool_usd_native"]
            frames.append(daily)
        except Exception as e:
            print(f"  skip {symbol_dir.name}: {e}")

    if not frames:
        return pd.DataFrame(
            columns=["date", "base", "pool_usd_total", "pool_usd_stable", "pool_usd_native"]
        )
    result = pd.concat(frames, ignore_index=True)
    return (
        result.groupby(["date", "base"])[["pool_usd_total", "pool_usd_stable", "pool_usd_native"]]
        .sum()
        .reset_index()
    )


print("Computing true pool USD for all GMX markets...")
gmx_pool_usd = load_gmx_pool_usd(GMX_POOL_DIR, price_monthly)
print(f"  {gmx_pool_usd['base'].nunique()} markets, {len(gmx_pool_usd)} daily rows")

# Sanity check — BTC pool: stable leg + WBTC priced
btc = gmx_pool_usd[gmx_pool_usd["base"] == "BTC"].sort_values("date").tail(3)
print("\nBTC pool USD (last 3 days):")
print(btc[["date", "pool_usd_stable", "pool_usd_native", "pool_usd_total"]].to_string(index=False))

Computing true pool USD for all GMX markets...
  113 markets, 45585 daily rows

BTC pool USD (last 3 days):
                     date  pool_usd_stable  pool_usd_native  pool_usd_total
2026-03-08 00:00:00+00:00     3.863249e+07              0.0    3.863249e+07
2026-03-09 00:00:00+00:00     3.835558e+07              0.0    3.835558e+07
2026-03-10 00:00:00+00:00     3.837892e+07              0.0    3.837892e+07


In [ ]:
# ── Comprehensive ranking: ALL GMX markets (OI + true pool USD) ──────────────
# Latest OI and pool USD per market
oi_all_latest = (
    gmx_oi_all.sort_values("date").groupby("base")["oi_usd"].last().reset_index(name="oi_latest")
)
pool_all_latest = (
    gmx_pool_usd.sort_values("date")
    .groupby("base")["pool_usd_total"]
    .last()
    .reset_index(name="pool_usd_latest")
)
oi_all_avg = gmx_oi_all.groupby("base")["oi_usd"].mean().reset_index(name="oi_avg")

# Longevity
lon_all = (
    gmx_oi_all.assign(date=lambda d: pd.to_datetime(d["date"]))
    .groupby("base")["date"]
    .agg(first_date="min", last_date="max")
    .reset_index()
)
lon_all["days_on_gmx"] = (lon_all["last_date"] - lon_all["first_date"]).dt.days

# Binance volume (where available)
binance_vol_avg = (
    binance_vol.groupby("base")[f"vol_ma{MA_WINDOW}d"].mean().reset_index(name="binance_vol_ma")
)

# Merge everything
gmx_universe = (
    oi_all_latest.merge(pool_all_latest, on="base", how="outer")
    .merge(oi_all_avg, on="base", how="left")
    .merge(lon_all[["base", "days_on_gmx", "first_date"]], on="base", how="left")
    .merge(binance_vol_avg, on="base", how="left")
    .fillna({"oi_latest": 0, "pool_usd_latest": 0, "oi_avg": 0})
)

# Rank (all markets, by OI + pool)
gmx_universe["rank_oi"] = gmx_universe["oi_latest"].rank(ascending=False, method="min")
gmx_universe["rank_pool"] = gmx_universe["pool_usd_latest"].rank(ascending=False, method="min")
gmx_universe["composite"] = (gmx_universe["rank_oi"] + gmx_universe["rank_pool"]) / 2
gmx_universe = gmx_universe.sort_values("composite").reset_index(drop=True)
gmx_universe["rank"] = range(1, len(gmx_universe) + 1)

# ── Print summary table ───────────────────────────────────────────────────────
disp_all = gmx_universe[
    [
        "rank",
        "base",
        "oi_latest",
        "pool_usd_latest",
        "oi_avg",
        "days_on_gmx",
        "first_date",
        "binance_vol_ma",
    ]
].copy()
disp_all.columns = [
    "#",
    "Base",
    "OI Latest ($M)",
    "Pool USD ($M)",
    "Avg OI ($M)",
    "GMX Days",
    "Since",
    "Binance Vol MA ($M/d)",
]
disp_all["OI Latest ($M)"] = (disp_all["OI Latest ($M)"] / 1e6).round(2)
disp_all["Pool USD ($M)"] = (disp_all["Pool USD ($M)"] / 1e6).round(2)
disp_all["Avg OI ($M)"] = (disp_all["Avg OI ($M)"] / 1e6).round(2)
disp_all["Binance Vol MA ($M/d)"] = (disp_all["Binance Vol MA ($M/d)"] / 1e6).round(1)
disp_all["Since"] = pd.to_datetime(disp_all["Since"]).dt.strftime("%Y-%m").fillna("N/A")

print(f"=== Full GMX Universe — {len(gmx_universe)} markets ===")
print(disp_all.to_string(index=False))

=== Full GMX Universe — 113 markets ===
  #                    Base  OI Latest ($M)  Pool USD ($M)  Avg OI ($M)  GMX Days   Since  Binance Vol MA ($M/d)
  1                     ETH           10.38          31.74        18.92     943.0 2023-08                12091.0
  2                     BTC            6.52          38.38        21.85     943.0 2023-08                17106.2
  3                     XRP            2.31           2.58         0.93     943.0 2023-08                    NaN
  4                     SOL            0.88           3.74         4.45     943.0 2023-08                 3589.8
  5                    LINK            1.37           3.19         2.97     943.0 2023-08                  359.9
  6                    HYPE            0.51           0.99         0.57     370.0 2025-03                    NaN
  7                    DOGE            0.20           1.36         0.91     943.0 2023-08                 1467.1
  8                     XMR            0.23           0.

In [ ]:
# ── Chart 1: Scatter — all markets, OI vs true pool USD ──────────────────────
scatter_all = gmx_universe[gmx_universe["pool_usd_latest"] > 0].copy()
scatter_all["oi_safe"] = scatter_all["oi_latest"].clip(lower=1e4)
scatter_all["has_binance"] = scatter_all["binance_vol_ma"].notna()

fig = px.scatter(
    scatter_all,
    x="pool_usd_latest",
    y="oi_latest",
    size="oi_safe",
    text="base",
    color="days_on_gmx",
    color_continuous_scale="Viridis",
    hover_data={"days_on_gmx": True, "oi_safe": False},
    title="All GMX Markets — Open Interest vs True Pool USD<br>"
    "<sup>Bubble size = OI  |  Colour = days of GMX data (darker = older market)</sup>",
    labels={
        "pool_usd_latest": "Pool USD — True (stable + native×price, $)",
        "oi_latest": "OI Latest (USD)",
        "days_on_gmx": "GMX History (days)",
    },
    template=TEMPLATE,
    size_max=55,
)
fig.update_traces(textposition="top center")
fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", height=650)
fig.show()

# ── Chart 2: Bar — top 30 markets by true pool USD ───────────────────────────
top30_pool = gmx_universe.sort_values("pool_usd_latest", ascending=False).head(30).copy()
top30_pool["pool_stable_M"] = top30_pool["pool_usd_latest"] / 1e6  # for display we'll use stacked
top30_pool_stable = (
    gmx_pool_usd.sort_values("date").groupby("base")["pool_usd_stable"].last().reset_index()
)
top30_pool_native = (
    gmx_pool_usd.sort_values("date").groupby("base")["pool_usd_native"].last().reset_index()
)
top30_pool = top30_pool.merge(top30_pool_stable, on="base", how="left").merge(
    top30_pool_native, on="base", how="left"
)

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        name="Stablecoin leg (USDC/USDT)",
        x=top30_pool["base"],
        y=top30_pool["pool_usd_stable"] / 1e6,
        marker_color="#9467bd",
    )
)
fig2.add_trace(
    go.Bar(
        name="Native token leg (priced via DeFiLlama)",
        x=top30_pool["base"],
        y=top30_pool["pool_usd_native"] / 1e6,
        marker_color="#1f77b4",
    )
)
fig2.update_layout(
    barmode="stack",
    title="Top 30 GMX Markets — True Pool USD Breakdown (Stable + Native × Price)",
    xaxis_title="Market",
    yaxis_title="Pool USD ($M)",
    template=TEMPLATE,
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=500,
    xaxis_tickangle=-45,
)
fig2.show()

In [ ]:
# ── Per-market detail: all GMX markets in groups of 15, true pool USD ────────
# Rerank by true pool USD (replaces stablecoin proxy used earlier)
pool_usd_rank = (
    gmx_pool_usd.groupby("base")["pool_usd_total"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
pool_usd_weekly = resample_weekly(
    gmx_pool_usd[["date", "base", "pool_usd_total"]].rename(
        columns={"pool_usd_total": "pool_usd_total"}
    ),
    "pool_usd_total",
)

# OI weekly already computed as gmx_oi_weekly
GROUP_SIZE_ALL = 15
oi_ranked_all = _oi_rank["base"].tolist()
oi_groups_all = [
    oi_ranked_all[i : i + GROUP_SIZE_ALL] for i in range(0, len(oi_ranked_all), GROUP_SIZE_ALL)
]
pool_groups_usd = [
    pool_usd_rank[i : i + GROUP_SIZE_ALL] for i in range(0, len(pool_usd_rank), GROUP_SIZE_ALL)
]

print(f"Total GMX markets in OI rank: {len(oi_ranked_all)}")
print(f"Total GMX markets in pool rank: {len(pool_usd_rank)}")
print(
    f"Groups of {GROUP_SIZE_ALL}: {len(oi_groups_all)} OI groups, {len(pool_groups_usd)} pool groups"
)
print()

# OI groups — weekly time series, all markets
for idx, group in enumerate(oi_groups_all):
    df_g = gmx_oi_weekly[gmx_oi_weekly["base"].isin(group)]
    fig = px.line(
        df_g,
        x="date",
        y="oi_usd",
        color="base",
        title=f"GMX Open Interest — Group {idx + 1}/{len(oi_groups_all)} "
        f"(OI rank {idx * GROUP_SIZE_ALL + 1}–{min((idx + 1) * GROUP_SIZE_ALL, len(oi_ranked_all))})",
        labels={"oi_usd": "OI (USD)", "date": "Date", "base": "Market"},
        template=TEMPLATE,
    )
    fig.update_layout(
        hovermode="x unified",
        yaxis_tickformat="$,.0f",
        height=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="v", x=1.01, y=1),
    )
    fig.show()

print("─" * 60)

# Pool USD groups — weekly time series, all markets
for idx, group in enumerate(pool_groups_usd):
    df_g = pool_usd_weekly[pool_usd_weekly["base"].isin(group)]
    if df_g.empty:
        continue
    fig = px.line(
        df_g,
        x="date",
        y="pool_usd_total",
        color="base",
        title=f"GMX Pool USD (True) — Group {idx + 1}/{len(pool_groups_usd)} "
        f"(pool rank {idx * GROUP_SIZE_ALL + 1}–{min((idx + 1) * GROUP_SIZE_ALL, len(pool_usd_rank))})",
        labels={
            "pool_usd_total": "Pool USD (stable + native×price)",
            "date": "Date",
            "base": "Market",
        },
        template=TEMPLATE,
    )
    fig.update_layout(
        hovermode="x unified",
        yaxis_tickformat="$,.0f",
        height=500,
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(orientation="v", x=1.01, y=1),
    )
    fig.show()

Total GMX markets in OI rank: 108
Total GMX markets in pool rank: 113
Groups of 15: 8 OI groups, 8 pool groups



────────────────────────────────────────────────────────────
